
# CIFAR-10 Image Classification using ANN and CNN

## Objective
The objective of this project is to build and compare different deep learning architectures for image classification on the CIFAR-10 dataset. The performance of a baseline Artificial Neural Network (ANN), a Convolutional Neural Network (CNN), and a Data-Augmented CNN is analyzed using validation accuracy and test accuracy metrics.


## 1. Import Required Libraries

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


## 2. Load and Preprocess the CIFAR-10 Dataset

In [ ]:

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Training Images Shape:", x_train.shape)
print("Testing Images Shape :", x_test.shape)

# Normalize pixel values
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

# Flatten images for ANN
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)


## 3. Baseline ANN Model

In [ ]:

ann_model = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

ann_history = ann_model.fit(
    x_train_flat,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=128,
    callbacks=[early_stop]
)

ann_test_loss, ann_test_acc = ann_model.evaluate(
    x_test_flat,
    y_test,
    verbose=0
)

print("ANN Test Accuracy:", ann_test_acc)


## 4. Convolutional Neural Network (CNN)

In [ ]:

cnn_model = models.Sequential([
    layers.Input(shape=(32,32,3)),

    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_history = cnn_model.fit(
    x_train_norm,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=128,
    callbacks=[early_stop]
)

cnn_test_loss, cnn_test_acc = cnn_model.evaluate(
    x_test_norm,
    y_test,
    verbose=0
)

print("CNN Test Accuracy:", cnn_test_acc)


## 5. Data Augmentation and Enhanced CNN

In [ ]:

augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_model = models.Sequential([
    layers.Input(shape=(32,32,3)),
    augmentation,

    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_history = aug_model.fit(
    x_train_norm,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=128,
    callbacks=[early_stop]
)

aug_test_loss, aug_test_acc = aug_model.evaluate(
    x_test_norm,
    y_test,
    verbose=0
)

print("Augmented CNN Test Accuracy:", aug_test_acc)


## 6. Validation Accuracy Comparison (ANN vs CNN)

In [ ]:

plt.figure(figsize=(10,5))

plt.plot(ann_history.history['val_accuracy'][:10],
         label='ANN')

plt.plot(cnn_history.history['val_accuracy'][:10],
         label='CNN')

plt.title('Validation Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## 7. Performance Comparison of All Models

In [ ]:

plt.figure(figsize=(10,5))

plt.plot(ann_history.history['val_accuracy'],
         label='ANN')

plt.plot(cnn_history.history['val_accuracy'],
         label='CNN')

plt.plot(aug_history.history['val_accuracy'],
         label='Augmented CNN')

plt.title('Validation Accuracy of All Models')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## 8. Test Accuracy Comparison

In [ ]:

results = pd.DataFrame({
    'Model':[
        'ANN',
        'CNN',
        'Augmented CNN'
    ],
    'Test Accuracy (%)':[
        round(ann_test_acc*100,2),
        round(cnn_test_acc*100,2),
        round(aug_test_acc*100,2)
    ]
})

results = results.sort_values(
    'Test Accuracy (%)',
    ascending=False
)

results



## 9. Conclusion

- The ANN model provides a baseline performance by treating images as flattened vectors.
- The CNN model achieves significantly better accuracy by learning spatial image features.
- Data augmentation improves the model's ability to generalize on unseen images.
- The augmented CNN achieves the best overall classification performance on the CIFAR-10 dataset.
